In [2]:
!pip install pandas cuid

  Preparing metadata (setup.py) ... done
  Created wheel for cuid: filename=cuid-0.4-py2.py3-none-any.whl size=4713 sha256=ff50e77e30f635612d631c60f9edd0ee2eddd129aa0c361651cbc9f031b9bd18
  Stored in directory: /root/.cache/pip/wheels/c1/9e/b8/eacbcd00d06e506913e04e7574775b2ddcb8fc78658823a0ca
Successfully built cuid


In [10]:
import pandas as pd
from cuid import cuid
import os

# Membaca file langsung dari direktori
file_path = 'data/fertilizer_recommendation.csv'

if os.path.exists(file_path):
    print(f"File found: {file_path}")
else:
    print(f"Warning: {file_path} not found. Please ensure the file is in the 'data' folder.")

File found: data/fertilizer_recommendation.csv


In [5]:
# Ganti 'dataset.csv' dengan nama file yang diupload
df = pd.read_csv('data/fertilizer_recommendation.csv')

# Cek kolom yang tersedia
print(df.columns)
print(df.isnull().sum()) # Identifikasi missing values

Index(['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Nitrogen_Level', 'Phosphorus_Level',
       'Potassium_Level', 'Temperature', 'Humidity', 'Rainfall', 'Crop_Type',
       'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Previous_Crop',
       'Region', 'Fertilizer_Used_Last_Season', 'Yield_Last_Season',
       'Recommended_Fertilizer'],
      dtype='object')
Soil_Type                      0
Soil_pH                        0
Soil_Moisture                  0
Organic_Carbon                 0
Electrical_Conductivity        0
Nitrogen_Level                 0
Phosphorus_Level               0
Potassium_Level                0
Temperature                    0
Humidity                       0
Rainfall                       0
Crop_Type                      0
Crop_Growth_Stage              0
Season                         0
Irrigation_Type                0
Previous_Crop                  0
Region                         0
Fertilizer_Used_Last_Sea

In [6]:
#Mapping SQL DUMP
mapping = {
    'Soil_Type': 'soil_type',
    'Temperature': 'temperature',
    'Humidity': 'humidity', # Pastikan ini yang dimaksud, bukan Soil_Moisture
    'Crop_Type': 'crop_type',
    'Nitrogen_Level': 'nitrogen',
    'Phosphorus_Level': 'phosphorus',
    'Potassium_Level': 'potassium',
    'Soil_pH': 'ph',
    'Organic_Carbon': 'organic_carbon',
    'Electrical_Conductivity': 'electrical_conductivity',
    'Rainfall': 'rainfall',
    'Recommended_Fertilizer': 'recommended_fertilizer'
}

# Rename & Select
df_clean = df.rename(columns=mapping)[list(mapping.values())].copy()

# Cleaning: Isi NULL dengan default aman atau drop baris
df_clean = df_clean.dropna()

# Casting tipe data
numeric_cols = ['temperature', 'humidity', 'nitrogen', 'phosphorus', 'potassium', 'ph', 'organic_carbon', 'electrical_conductivity', 'rainfall']
df_clean[numeric_cols] = df_clean[numeric_cols].astype(float)

In [7]:
# Generate ID Unik
df_clean['id'] = [cuid() for _ in range(len(df_clean))]

# Fungsi escape single quote untuk mencegah SQL error
def escape_sql(val):
    if isinstance(val, str):
        return val.replace("'", "''")
    return val

# Generate SQL Dump
sql_commands = ["BEGIN;"]
for _, row in df_clean.iterrows():
    values = [
        f"'{escape_sql(row['id'])}'",
        f"'{escape_sql(row['soil_type'])}'",
        f"{row['temperature']}",
        f"{row['humidity']}",
        f"'{escape_sql(row['crop_type'])}'" if pd.notnull(row['crop_type']) else "NULL",
        f"{row['nitrogen']}",
        f"{row['phosphorus']}",
        f"{row['potassium']}",
        f"{row['ph']}",
        f"{row['organic_carbon']}" if pd.notnull(row['organic_carbon']) else "NULL",
        f"{row['electrical_conductivity']}" if pd.notnull(row['electrical_conductivity']) else "NULL",
        f"{row['rainfall']}",
        f"'{escape_sql(row['recommended_fertilizer'])}'"
    ]
    sql = f"INSERT INTO \"soil_references\" (\"id\", \"soil_type\", \"temperature\", \"humidity\", \"crop_type\", \"nitrogen\", \"phosphorus\", \"potassium\", \"ph\", \"organic_carbon\", \"electrical_conductivity\", \"rainfall\", \"recommended_fertilizer\") VALUES ({', '.join(values)});"
    sql_commands.append(sql)

sql_commands.append("COMMIT;")
final_sql = "\n".join(sql_commands)

In [8]:
import os

# Pastikan direktori sql_dump ada
os.makedirs('sql_dump', exist_ok=True)

file_path = 'sql_dump/seed_data.sql'
with open(file_path, 'w') as f:
    f.write(final_sql)

# Cek apakah sedang di Google Colab untuk download otomatis
try:
    from google.colab import files
    files.download(file_path)
except ImportError:
    print(f"File berhasil disimpan di: {os.path.abspath(file_path)}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>